# Principal Component Analysis (PCA) for Accident Quantities & Severity
**Dataset**: `new 2000 records/US_Accidents_2000_sample.csv` (strictly 2,000 sample records)

### Objectives:
1. **State-Level PCA**: Analyze the multidimensional relationship between accident quantities (overall volume, severity-specific counts) and severity metrics (mean severity, high-severity rate).
2. **PCA Loadings & Correlation Heatmaps**: Visualize component weights and correlations in heatmaps.
3. **Temporal Heatmaps**: Visualize accidents by hour of day and weekday across both **Quantity** and **Severity** dimensions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Visual styling
sns.set_theme(style="white", font="sans-serif")
plt.rcParams["figure.autolayout"] = True

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Strictly load from US_Accidents_2000_sample.csv
DATA_PATH = os.path.join("new 2000 records", "US_Accidents_2000_sample.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = "US_Accidents_2000_sample.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} records across {df['State'].nunique()} states.")

## 1. Feature Engineering: Accident Quantities & Severity Metrics by State
For each geographic entity (State), we calculate:
- **Total Accidents**: Overall accident frequency/quantity.
- **Severity 1..4 Quantities**: Breakdown count of incidents at each severity level.
- **Mean Severity**: Weighted average severity score (1.0 to 4.0).
- **High Severity Rate (%)**: Proportion of accidents classified as Severity 3 or 4.

In [ ]:
state_profile = df.groupby("State", observed=True).agg(
    Total_Accidents=("Severity", "count"),
    Severity_1_Qty=("Severity", lambda s: (s == 1).sum()),
    Severity_2_Qty=("Severity", lambda s: (s == 2).sum()),
    Severity_3_Qty=("Severity", lambda s: (s == 3).sum()),
    Severity_4_Qty=("Severity", lambda s: (s == 4).sum()),
    Mean_Severity=("Severity", "mean"),
    High_Severity_Pct=("Severity", lambda s: (s >= 3).mean() * 100)
).reset_index()

feature_cols = [
    "Total_Accidents", "Severity_1_Qty", "Severity_2_Qty",
    "Severity_3_Qty", "Severity_4_Qty", "Mean_Severity", "High_Severity_Pct"
]

feature_labels = [
    "Total Accidents", "Severity 1 Qty", "Severity 2 Qty",
    "Severity 3 Qty", "Severity 4 Qty", "Mean Severity", "High Severity Rate (%)"
]

state_profile.head(10)

## 2. Standardization & Principal Component Analysis (PCA)
Since quantities (e.g. counts in hundreds) and severity rates (means and percentages) have different scales, we standardize to mean=0 and unit variance before running PCA.

In [ ]:
X = state_profile[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

explained_var = pca.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

print("PCA Explained Variance:")
for i, (ev, cv) in enumerate(zip(explained_var, cum_var)):
    print(f"  PC{i+1}: {ev*100:5.2f}% (Cumulative: {cv*100:5.2f}%)")

loadings = pd.DataFrame(
    pca.components_,
    columns=feature_labels,
    index=[f"PC{i+1} ({ev*100:.1f}%)" for i, ev in enumerate(explained_var)]
)
loadings.round(3)

## 3. Heatmap 1: PCA Component Loadings Heatmap
- **PC1 (53.2% of variance)**: High positive weights for **Total Accidents**, **Severity 2 Qty**, and **Severity 3 Qty** -> **Accident Quantity / Volume Dimension**.
- **PC2 (32.6% of variance)**: High positive weights for **Mean Severity** (+0.650) and **High Severity Rate** (+0.630) -> **Accident Severity Dimension**.
- Together, **PC1 + PC2 capture 85.8% of the total variance**!

In [ ]:
plt.figure(figsize=(11, 5.5))
sns.heatmap(
    loadings.iloc[:4],
    annot=True,
    fmt=".3f",
    cmap="vlag",
    center=0,
    vmin=-0.8,
    vmax=0.8,
    cbar_kws={"label": "Component Loading (Weight)", "shrink": 0.8},
    linewidths=1.2,
    linecolor="white"
)

plt.title(
    "PCA Component Loadings Heatmap: Accident Quantities & Severity\n" \
    "PC1 = Accident Quantity Dimension (53.2%) | PC2 = Accident Severity Dimension (32.6%)",
    fontsize=12, fontweight="bold", pad=15
)
plt.xlabel("Accident Quantity & Severity Features", fontsize=11, fontweight="bold", labelpad=10)
plt.ylabel("Principal Components", fontsize=11, fontweight="bold", labelpad=10)
plt.xticks(rotation=20, ha="right")
plt.yticks(rotation=0)
plt.savefig("figures/pca_loadings_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Heatmap 2: Feature-to-Component Correlation Heatmap
Shows Pearson correlation between each original metric and each principal component.

In [ ]:
eigenvalues = pca.explained_variance_
component_correlations = pd.DataFrame(
    pca.components_[:4].T * np.sqrt(eigenvalues[:4]),
    index=feature_labels,
    columns=[f"PC{i+1} ({ev*100:.1f}%)" for i, ev in enumerate(explained_var[:4])]
)

plt.figure(figsize=(8, 6.5))
sns.heatmap(
    component_correlations,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    vmin=-1.0,
    vmax=1.0,
    cbar_kws={"label": "Pearson Correlation (r)", "shrink": 0.8},
    linewidths=1.2,
    linecolor="white"
)

plt.title("Correlation Heatmap: Features vs Principal Components", fontsize=12, fontweight="bold", pad=15)
plt.xlabel("Principal Components", fontsize=11, fontweight="bold", labelpad=10)
plt.ylabel("Original Metrics", fontsize=11, fontweight="bold", labelpad=10)
plt.savefig("figures/pca_feature_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Temporal Heatmap: Weekday x Hour by Severity (Quantity vs. Severity Dimensions)
Comparing accident volume against mean severity across all 168 hours of the week.

In [ ]:
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
df["Hour"] = df["Start_Time"].dt.hour
df["Weekday"] = df["Start_Time"].dt.day_name()
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Mean Severity Heatmap across Weekday x Hour (colormap: plasma)
sw = df.pivot_table(index="Weekday", columns="Hour", values="Severity", aggfunc="mean").reindex(WEEKDAYS)

fig, ax = plt.subplots(figsize=(15, 4.5))
sns.heatmap(sw, cmap="plasma", linewidths=0.4, linecolor="white", annot=True, fmt=".2f",
            annot_kws={"size": 7}, cbar_kws={"label": "Mean Severity"}, ax=ax)
ax.set_title("Mean severity by hour of day and weekday", fontsize=12, fontweight="bold")
ax.set_ylabel("")
ax.set_xlabel("Hour")
plt.savefig("figures/mean_severity_hour_weekday_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Stacked Dimension Comparison: Quantity Heatmap vs. Severity Heatmap

In [ ]:
hw = pd.crosstab(df["Weekday"], df["Hour"]).reindex(WEEKDAYS)

fig, axes = plt.subplots(2, 1, figsize=(15, 9.5))
sns.heatmap(hw, cmap="YlOrRd", linewidths=0.4, linecolor="white", annot=True, fmt="d",
            annot_kws={"size": 7}, cbar_kws={"label": "Accidents"}, ax=axes[0])
axes[0].set_title("Accidents by hour of day and weekday (Quantity Dimension)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("")
axes[0].set_xlabel("Hour")

sns.heatmap(sw, cmap="plasma", linewidths=0.4, linecolor="white", annot=True, fmt=".2f",
            annot_kws={"size": 7}, cbar_kws={"label": "Mean Severity"}, ax=axes[1])
axes[1].set_title("Mean severity by hour of day and weekday (Severity Dimension)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("")
axes[1].set_xlabel("Hour")

plt.savefig("figures/accidents_and_severity_hour_weekday_stacked.png", dpi=300, bbox_inches="tight")
plt.show()